In [2]:
# FinFET Modeling using ML + PAL Optimization

In [3]:
# Importing Required Libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
import ipywidgets as widgets
from IPython.display import display

In [4]:
# Dataset Generation and Device Modeling (TCAD-Inspired)
np.random.seed(0)
n = 600

data = pd.DataFrame({
    'Lg': np.random.uniform(10,50,n),
    'Wfin': np.random.uniform(5,20,n),
    'Hfin': np.random.uniform(20,60,n),
    'Tox': np.random.uniform(1,5,n),
    'Vgs': np.random.uniform(0.3,1.0,n),
    'Vds': np.random.uniform(0.3,1.0,n),
    'Doping': np.random.uniform(1e16,1e18,n)
})

# Outputs (device physics inspired)
data['Id'] = (data['Wfin']*data['Hfin']*data['Vgs']*data['Vds'])/(data['Lg']*data['Tox']) * np.log(data['Doping'])

data['Delay'] = data['Lg']/(data['Vgs']*data['Wfin'])

data['Power'] = (data['Vds']**2 * data['Wfin'])/data['Tox']

data['Vth'] = data['Tox']*0.2 + 0.3
data['Ion'] = data['Id']*1.2
data['Ioff'] = data['Id']*0.01

print(" FULL DATASET")
display(data)

print("Shape:", data.shape)

 FULL DATASET


,Lg,Wfin,Hfin,Tox,Vgs,Vds,Doping,Id,Delay,Power,Vth,Ion,Ioff
0,31.952540,7.619876,32.341118,3.187800,0.920678,0.589774,2.648918e+17,52.703577,4.554593,0.831434,0.937560,63.244292,0.527036
1,38.607575,9.919820,57.687389,2.174467,0.881636,0.740733,9.590085e+17,184.314561,4.414478,2.503076,0.734893,221.177474,1.843146
2,34.110535,15.205230,55.530602,4.872818,0.322124,0.845009,8.198768e+17,57.035205,6.964224,2.228104,1.274564,68.442246,0.570352
3,31.795327,5.948114,54.412427,1.904785,0.697893,0.896090,7.269863e+17,137.449526,7.659404,2.507477,0.680957,164.939431,1.374495
4,26.946192,14.108741,46.119990,1.062953,0.726413,0.871489,4.671937e+17,585.128546,2.629212,10.080868,0.512591,702.154255,5.851285
...,...,...,...,...,...,...,...,...,...,...,...,...,...
595,49.918490,13.406759,56.199293,1.744846,0.959181,0.586182,5.192965e+17,198.397475,3.881835,2.640169,0.648969,238.076970,1.983975
596,24.487562,15.023273,49.948892,1.753598,0.682293,0.381336,1.228078e+17,178.908913,2.388967,1.245807,0.650720,214.690696,1.789089
597,28.825958,9.300750,42.444837,2.743365,0.526530,0.974166,5.082149e+17,104.392282,5.886308,3.217362,0.848673,125.270738,1.043923
598,25.129807,5.291937,53.461887,3.954369,0.869482,0.981635,4.385700e+17,98.711448,5.461528,1.289548,1.090874,118.453737,0.987114


Shape: (600, 13)


In [ ]:

#plotting
plt.figure()
plt.scatter(data['Delay'], data['Power'])
plt.xlabel("Delay")
plt.ylabel("Power")
plt.title("Initial Design Space (Before Optimization)")
plt.show()

In [7]:
# Machine Learning Model Training (Regression)
X = data[['Lg','Wfin','Hfin','Tox','Vgs','Vds','Doping']]
y_Id = data['Id']
y_delay = data['Delay']
y_power = data['Power']

X_train, X_test = train_test_split(X, test_size=0.2)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)

model_Id = RandomForestRegressor()
model_delay = RandomForestRegressor()
model_power = RandomForestRegressor()

model_Id.fit(X_train, y_Id[:len(X_train)])
model_delay.fit(X_train, y_delay[:len(X_train)])
model_power.fit(X_train, y_power[:len(X_train)])

print(" Models trained")

 Models trained


In [6]:
X_scaled = scaler.transform(X)

data['Predicted_Id'] = model_Id.predict(X_scaled)
data['Predicted_Delay'] = model_delay.predict(X_scaled)
data['Predicted_Power'] = model_power.predict(X_scaled)

NameError: name 'scaler' is not defined

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
import ipywidgets as widgets
from IPython.display import display

In [ ]:
## Interactive User Interface (Sliders & Control Panel)
import ipywidgets as widgets
from IPython.display import display

# Sliders
Lg_s = widgets.FloatSlider(value=25, min=10, max=50, step=1, description='Lg (nm)')
Wfin_s = widgets.FloatSlider(value=10, min=5, max=20, step=1, description='Wfin (nm)')
Hfin_s = widgets.FloatSlider(value=40, min=20, max=60, step=1, description='Hfin (nm)')
Tox_s = widgets.FloatSlider(value=2, min=1, max=5, step=0.1, description='Tox (nm)')
Vgs_s = widgets.FloatSlider(value=0.7, min=0.3, max=1.0, step=0.05, description='Vgs (V)')
Vds_s = widgets.FloatSlider(value=0.7, min=0.3, max=1.0, step=0.05, description='Vds (V)')
Doping_s = widgets.FloatSlider(value=1e17, min=1e16, max=1e18, step=1e16, description='Doping')

# Layout
left = widgets.VBox([Lg_s, Wfin_s, Hfin_s])
right = widgets.VBox([Tox_s, Vgs_s, Vds_s, Doping_s])
ui = widgets.HBox([left, right])

# Button + Output
button = widgets.Button(description="Run Optimization", button_style='success')
output = widgets.Output()

# Display order (IMPORTANT)
display(ui)
display(button, output) 

In [ ]:
print("Doping:", "{:.2e}".format(Doping_s.value))

In [ ]:
## Model Prediction and PAL-Based Optimization
def run_model(b):
    with output:
        output.clear_output(wait=True)

        # 1. Take input from sliders
        user_input = pd.DataFrame({
            'Lg':[Lg_s.value],
            'Wfin':[Wfin_s.value],
            'Hfin':[Hfin_s.value],
            'Tox':[Tox_s.value],
            'Vgs':[Vgs_s.value],
            'Vds':[Vds_s.value],
            'Doping':[Doping_s.value]
        })

        # Scale input
        scaled = scaler.transform(user_input)

        # -------------------------
        #  BEFORE OPTIMIZATION
        # -------------------------
        pred_Id = model_Id.predict(scaled)[0]
        pred_delay = model_delay.predict(scaled)[0]
        pred_power = model_power.predict(scaled)[0]

        print(" BEFORE OPTIMIZATION")
        print("Id    :", round(pred_Id,4))
        print("Delay :", round(pred_delay,4))
        print("Power :", round(pred_power,4))

        # Graph (Before)
        plt.figure()
        plt.scatter(data['Delay'], data['Power'], alpha=0.3)
        plt.scatter(pred_delay, pred_power, marker='X')
        plt.xlabel("Delay")
        plt.ylabel("Power")
        plt.title("Before Optimization")
        plt.show()

        # -------------------------
        #  PAL OPTIMIZATION
        # -------------------------
        delay_vals = []
        power_vals = []
        Id_vals = []

        for i in range(150):
            temp = user_input.copy()

            # Small variations (PAL idea)
            temp['Lg'] += np.random.uniform(-2,2)
            temp['Wfin'] += np.random.uniform(-1,1)
            temp['Hfin'] += np.random.uniform(-2,2)
            temp['Tox'] += np.random.uniform(-0.2,0.2)
            temp['Vgs'] += np.random.uniform(-0.05,0.05)
            temp['Vds'] += np.random.uniform(-0.05,0.05)
            temp['Doping'] += np.random.uniform(-1e16,1e16)

            temp_scaled = scaler.transform(temp)

            d = model_delay.predict(temp_scaled)[0]
            p = model_power.predict(temp_scaled)[0]
            i_val = model_Id.predict(temp_scaled)[0]

            delay_vals.append(d)
            power_vals.append(p)
            Id_vals.append(i_val)

        # Best design (minimum power)
        best_idx = np.argmin(power_vals)

        best_delay = delay_vals[best_idx]
        best_power = power_vals[best_idx]
        best_Id = Id_vals[best_idx]

        print("\n AFTER OPTIMIZATION")
        print("Id    :", round(best_Id,4))
        print("Delay :", round(best_delay,4))
        print("Power :", round(best_power,4))

        # Graph (After)
        plt.figure()
        plt.scatter(delay_vals, power_vals, alpha=0.5)
        plt.scatter(pred_delay, pred_power, marker='X', label="Before")
        plt.scatter(best_delay, best_power, marker='*', s=200, label="Best")

        plt.xlabel("Delay")
        plt.ylabel("Power")
        plt.title("After Optimization (PAL)")
        plt.legend()
        plt.show()

        # Comparison
        print("\n COMPARISON")
        print("Before → Id:", round(pred_Id,4),
              " Delay:", round(pred_delay,4),
              " Power:", round(pred_power,4))

        print("After  → Id:", round(best_Id,4),
              " Delay:", round(best_delay,4),
              " Power:", round(best_power,4))

In [ ]:
button.on_click(run_model)